# Titanic 3: Title
이번 가설: 이름 원문은 고차원이지만 호칭은 성별·연령과 관련된 낮은 차원의 정보를 보존할 수 있다.
이전까지 채택된 추가 피처: 없음.
이번 변경: **Title 하나 추가**, 다른 모델·하이퍼파라미터 변경 없음. HasCabin/AgeMissing 제외 유지.

채택 규칙(평가 전 고정): 이전 Best보다 holdout AUC 상승 → 동일 5-fold 비교 → CV 평균 상승 및 Gap 증가 0.01 이내이면 채택. Gap 0.01은 운영 기준이며 통계적 유의성 기준이 아니다.
CV도 이 데이터로 후보를 고르는 보조 검증이므로 반복 선택의 낙관성이 남는다. 여러 CV fold 결과로 새 threshold를 탐색하지 않는다.
CSV는 해당 버전에서 **실제로 실험한 후보 피처 집합**의 예측을 보존한다. 탈락 피처는 다음 버전의 입력에 누적하지 않는다. 최종 채택 Best와 실험 CSV를 구분한다.

# 1. 데이터 불러오기

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from common.utils import train_test_split_by_target
from common.feature_experiments import FeaturePreprocessor, fit_cat, scores, cv_compare, baseline_parameters

train = pd.read_csv('csv/train.csv')
test = pd.read_csv('csv/test.csv')
submission = pd.read_csv('csv/submission.csv')
previous_features = []
new_feature = 'Title'
candidate_features = previous_features + [new_feature]
previous_cv = {'mean': 0.9017752715121137, 'std': 0.016321824128332087}
print(train.shape, test.shape)
display(train.head())
display(test.head())

,passengerid,survived,pclass,name,gender,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,0,2,"Wheeler, Mr. Edwin Frederick""""",male,NaN,0,0,SC/PARIS 2159,12.8750,NaN,S
1,1,0,3,"Henry, Miss. Delia",female,NaN,0,0,382649,7.7500,NaN,Q
2,2,1,1,"Hays, Mrs. Charles Melville (Clara Jennings Gr...",female,52.0,1,1,12749,93.5000,B69,S
3,3,1,3,"Andersson, Mr. August Edvard (""Wennerstrom"")",male,27.0,0,0,350043,7.7958,NaN,S
4,4,0,2,"Hold, Mr. Stephen",male,44.0,1,0,26707,26.0000,NaN,S


,passengerid,pclass,name,gender,age,sibsp,parch,ticket,fare,cabin,embarked
0,916,3,"McGowan, Miss. Anna ""Annie""",female,15.0,0,0,330923,8.0292,NaN,Q
1,917,2,"Pinsky, Mrs. (Rosa)",female,32.0,0,0,234604,13.0000,NaN,S
2,918,3,"McCarthy, Miss. Catherine Katie""""",female,NaN,0,0,383123,7.7500,NaN,Q
3,919,3,"Franklin, Mr. Charles (Charles Fardon)",male,NaN,0,0,SOTON/O.Q. 3101314,7.2500,NaN,S
4,920,1,"Wick, Mrs. George Dennick (Mary Hitchcock)",female,45.0,1,1,36928,164.8667,NaN,S


(916, 12) (393, 11)


# 2. 데이터 메타 정보 확인

In [2]:
train.info()
print(train.isna().sum())
print(train.nunique())
test.info()
print(test.isna().sum())
print('train에만:', set(train.columns)-set(test.columns))

<class 'pandas.DataFrame'>
RangeIndex: 916 entries, 0 to 915
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  916 non-null    int64  
 1   survived     916 non-null    int64  
 2   pclass       916 non-null    int64  
 3   name         916 non-null    str    
 4   gender       916 non-null    str    
 5   age          736 non-null    float64
 6   sibsp        916 non-null    int64  
 7   parch        916 non-null    int64  
 8   ticket       916 non-null    str    
 9   fare         916 non-null    float64
 10  cabin        198 non-null    str    
 11  embarked     915 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 86.0 KB
passengerid      0
survived         0
pclass           0
name             0
gender           0
age            180
sibsp            0
parch            0
ticket           0
fare             0
cabin          718
embarked         1
dtype: int64
passengerid    916

# 3. Target 정의

In [3]:
assert set(train.columns)-set(test.columns) == {'survived'}
assert set(test.columns)-set(train.columns) == set()
target_col = 'survived'
id_col = 'passengerid'
X = train.drop(columns=[target_col, id_col])
y = train[target_col]
X_test_raw = test.drop(columns=id_col)
assert X.columns.equals(X_test_raw.columns)
assert train[id_col].is_unique and test[id_col].is_unique
assert set(train[id_col]).isdisjoint(test[id_col])

# 4. Target 분석
Baseline과 같은 seed 42, stratify, validation 25%를 유지합니다.

In [4]:
print(y.value_counts(normalize=True))
train_part, valid_part = train_test_split_by_target(train, target_name=target_col)
X_tr_raw = train_part.drop(columns=[target_col, id_col])
y_tr = train_part[target_col]
X_valid_raw = valid_part.drop(columns=[target_col, id_col])
y_valid = valid_part[target_col]
assert len(train_part)==687 and len(valid_part)==229
print('학습/검증:', X_tr_raw.shape, X_valid_raw.shape)

survived
0    0.622271
1    0.377729
Name: proportion, dtype: float64
학습/검증: (687, 10) (229, 10)


# 5. EDA
학습 부분만 사용해 후보를 임시 계산하고 target과의 관계를 확인합니다. 모델 입력에 채택하기 전의 점검입니다.
생성 방법 / 사용 원본: name.str.extract(r',\s*([^.]*)\.') → strip → 학습 빈도 5 이하/미관측은 Rare. 학습 EDA에서 주요 4개는 36명 이상, 나머지는 5명 이하여 이 경계를 선택했다.
validation/test는 빈도·구간·중앙값 결정에 사용하지 않습니다.

In [5]:
eda_prep = FeaturePreprocessor(candidate_features).fit_missing(X_tr_raw)
eda_frame = eda_prep.feature_frame(X_tr_raw, eda_prep.transform_missing(X_tr_raw))
eda_frame[target_col] = y_tr
raw_titles = eda_prep.titles(X_tr_raw)
display(pd.DataFrame({'Title':raw_titles,'survived':y_tr}).groupby('Title')['survived'].agg(['count','mean']))
display(pd.crosstab(eda_frame['Title'], X_tr_raw['gender']))
display(eda_frame.groupby('Title')['age'].agg(['count','median']))
display(eda_frame.groupby(new_feature,dropna=False)[target_col].agg(['count','mean']))
print('EDA 표본 수:', len(eda_frame))

,count,mean
Title,,
Capt,1,0.000000
Col,3,0.000000
Don,1,0.000000
Dr,4,0.250000
Jonkheer,1,0.000000
Major,2,0.500000
Master,36,0.333333
Miss,139,0.827338
Mr,393,0.101781


gender,female,male
Title,,
Master,0,36
Miss,139,0
Mr,0,393
Mrs,100,0
Rare,1,18


,count,median
Title,,
Master,36,6.0
Miss,139,22.0
Mr,393,27.0
Mrs,100,33.0
Rare,19,45.0


,count,mean
Title,,
Master,36,0.333333
Miss,139,0.827338
Mr,393,0.101781
Mrs,100,0.890000
Rare,19,0.210526


EDA 표본 수: 687


해석 기준: gender·age와 정보가 겹치므로 성별 교차표와 호칭별 연령 통계를 함께 확인한다. 원본 성별/나이만으로 설명되지 않는 정보가 있는지는 AUC로 판단한다. Rare의 생존율은 작은 표본을 합친 값이다. 숫자형의 Pearson 상관만으로 삭제하지 않습니다. 카테고리 표의 count가 작을수록 평균 생존율을 조심해서 해석합니다.

# 6. 데이터 전처리
## 6-1. 결측치
Baseline의 나이 그룹 중앙값, 전체 중앙값 fallback, 수치형 중앙값/범주형 최빈값과 cabin 원문 삭제를 재사용합니다. 원본 X는 보존하여 이름/티켓/객실 파생변수에 사용합니다.

In [6]:
prep = FeaturePreprocessor(candidate_features).fit_missing(X_tr_raw)
X_tr_clean = prep.transform_missing(X_tr_raw)
X_valid_clean = prep.transform_missing(X_valid_raw)
assert not X_tr_clean.isna().any().any()
assert not X_valid_clean.isna().any().any()

## 6-2. Feature 생성
Baseline FamilySize/IsAlone + 이전 채택 피처 + 이번 Title만 생성합니다. 실제 생성 함수는 새 common/feature_experiments.py에 있고 기존 common은 그대로입니다.

In [7]:
X_tr_features = prep.feature_frame(X_tr_raw, X_tr_clean)
X_valid_features = prep.feature_frame(X_valid_raw, X_valid_clean)
print('추가 피처:', candidate_features)
print('인코딩 전:', X_tr_features.columns.tolist())
display(X_tr_features.head())

,pclass,gender,age,sibsp,parch,fare,embarked,FamilySize,IsAlone,Title
595,3,male,0.83,0,1,9.3500,S,2,0,Master
148,3,male,24.00,0,0,8.0500,S,1,1,Mr
347,3,female,9.00,1,1,15.2458,C,3,0,Miss
731,3,female,24.00,0,2,16.7000,S,3,0,Mrs
457,2,male,44.00,0,0,13.0000,S,1,1,Mr


추가 피처: ['Title']
인코딩 전: ['pclass', 'gender', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'FamilySize', 'IsAlone', 'Title']


## 6-3. 중복/Feature 관계 확인
이름/티켓 원문은 baseline처럼 제외합니다. 파생 정보가 있으면 그 피처만 남습니다. Row 중복은 baseline처럼 검증만 하며 승객을 제거하지 않습니다.

In [8]:
assert train.duplicated().sum()==0
assert train.drop(columns=id_col).duplicated().sum()==0
if pd.api.types.is_numeric_dtype(X_tr_features[new_feature]):
    numeric = X_tr_features.select_dtypes(include='number')
    display(numeric.corr()[new_feature].sort_values())
else:
    display(pd.crosstab(X_tr_features[new_feature], X_tr_raw['pclass']))
    display(pd.crosstab(X_tr_features[new_feature], X_tr_raw['gender']))

pclass,1,2,3
Title,,,
Master,3,6,27
Miss,29,28,82
Mr,84,88,221
Mrs,36,31,33
Rare,13,6,0


gender,female,male
Title,,
Master,0,36
Miss,139,0
Mr,0,393
Mrs,100,0
Rare,1,18


gender·age와 정보가 겹치므로 성별 교차표와 호칭별 연령 통계를 함께 확인한다. 원본 성별/나이만으로 설명되지 않는 정보가 있는지는 AUC로 판단한다. Rare의 생존율은 작은 표본을 합친 값이다.

## 6-4. Encoding
학습 부분에서만 OneHotEncoder를 fit하고 validation에는 transform만 합니다. 새로운 범주형도 같은 방식, 미관측 범주는 ignore입니다.

In [9]:
prep.fit_encoding(X_tr_features)
X_tr_model = prep.transform_encoding(X_tr_features)
X_valid_model = prep.transform_encoding(X_valid_features)
assert X_tr_model.columns.equals(X_valid_model.columns)
assert X_tr_model.index.equals(y_tr.index)
print(X_tr_model.shape, X_valid_model.shape)

(687, 17) (229, 17)


## 6-5. Scaling
Baseline과 동일하게 적용하지 않습니다.

In [10]:
assert np.isfinite(X_tr_model.to_numpy()).all()
assert np.isfinite(X_valid_model.to_numpy()).all()

# 7. 모델 학습
원본 baseline, 이전 Best, 후보를 동일한 모델로 학습합니다. 원본 baseline의 실제 파라미터 전체와 후보를 비교합니다. learning_rate 등의 자동 기본값도 같은 학습 크기에서 동일해야 합니다.

In [11]:
reference_prep = FeaturePreprocessor()
reference_X = reference_prep.fit_transform(X_tr_raw)
reference_valid = reference_prep.transform(X_valid_raw)
reference_model = fit_cat(reference_X, y_tr)
reference_scores = scores(reference_model, reference_X, y_tr, reference_valid, y_valid)
assert abs(reference_scores['validation_auc'] - 0.900065) < 0.000001
previous_prep = FeaturePreprocessor(previous_features)
previous_X = previous_prep.fit_transform(X_tr_raw)
previous_valid = previous_prep.transform(X_valid_raw)
previous_model = reference_model if not previous_features else fit_cat(previous_X, y_tr)
previous_scores = scores(previous_model, previous_X, y_tr, previous_valid, y_valid)
candidate_model = fit_cat(X_tr_model, y_tr)
assert candidate_model.get_all_params() == reference_model.get_all_params()
assert previous_model.get_all_params() == reference_model.get_all_params()
print('명시적 설정:', baseline_parameters())
print('실효 설정:', candidate_model.get_all_params())

명시적 설정: {'verbose': 0, 'random_state': 42, 'cat_features': [], 'allow_writing_files': False}
실효 설정: {'nan_mode': 'Min', 'eval_metric': 'Logloss', 'iterations': 1000, 'sampling_frequency': 'PerTree', 'leaf_estimation_method': 'Newton', 'random_score_type': 'NormalWithModelSizeDecrease', 'grow_policy': 'SymmetricTree', 'penalties_coefficient': 1, 'boosting_type': 'Plain', 'model_shrink_mode': 'Constant', 'feature_border_type': 'GreedyLogSum', 'bayesian_matrix_reg': 0.10000000149011612, 'eval_fraction': 0, 'force_unit_auto_pair_weights': False, 'l2_leaf_reg': 3, 'random_strength': 1, 'rsm': 1, 'boost_from_average': False, 'model_size_reg': 0.5, 'pool_metainfo_options': {'tags': {}}, 'subsample': 0.800000011920929, 'use_best_model': False, 'class_names': [0, 1], 'random_seed': 42, 'depth': 6, 'posterior_sampling': False, 'border_count': 254, 'classes_count': 0, 'auto_class_weights': 'None', 'sparse_features_conflict_fraction': 0, 'leaf_estimation_backtracking': 'AnyImprovement', 'best_mode

# 8. Evaluation
양성 클래스 확률로 AUC를 계산합니다. 이전 Best 대비 상승 후보만 5-fold를 수행합니다.

In [12]:
candidate_scores = scores(candidate_model, X_tr_model, y_tr, X_valid_model, y_valid)
holdout_delta = candidate_scores['validation_auc'] - previous_scores['validation_auc']
original_delta = candidate_scores['validation_auc'] - reference_scores['validation_auc']
display(pd.DataFrame({'Original baseline':reference_scores, 'Previous best':previous_scores, 'Candidate':candidate_scores}))
print('이전 Best 대비:', holdout_delta, '최초 baseline 대비:', original_delta)
candidate_cv = None
cv_folds = None
if holdout_delta > 0:
    cv_folds, cv_table = cv_compare(X, y, previous_features, candidate_features)
    display(cv_folds.pivot(index='fold', columns='model', values='auc'))
    display(cv_table)
    previous_cv = cv_table.loc['Previous best'].to_dict()
    candidate_cv = cv_table.loc['Candidate'].to_dict()
accepted = (holdout_delta > 0 and candidate_cv is not None
            and candidate_cv['mean'] > previous_cv['mean']
            and candidate_scores['gap'] <= previous_scores['gap'] + 0.01)
best_features = candidate_features if accepted else previous_features
best_scores = candidate_scores if accepted else previous_scores
best_cv = candidate_cv if accepted else previous_cv
print('채택:', accepted, '다음 버전의 Best:', best_features)

,Original baseline,Previous best,Candidate
train_auc,0.963079,0.963079,0.965168
validation_auc,0.900065,0.900065,0.902992
gap,0.063014,0.063014,0.062176


model,Candidate,Original baseline,Previous best
fold,,,
1,0.891416,0.891917,0.891917
2,0.913933,0.919082,0.919082
3,0.875985,0.878782,0.878782
4,0.906051,0.906433,0.906433
5,0.912662,0.912662,0.912662


,mean,std
model,,
Candidate,0.900010,0.016136
Original baseline,0.901775,0.016322
Previous best,0.901775,0.016322


이전 Best 대비: 0.0029273052528866472 최초 baseline 대비: 0.0029273052528866472
Fold 1 완료
Fold 2 완료
Fold 3 완료
Fold 4 완료
Fold 5 완료
채택: False 다음 버전의 Best: []


# 9. 전체 Train 재학습
요청한 버전별 CSV는 채택 여부와 무관하게 이번에 실험한 후보 피처로 생성합니다. 다음 실험은 채택된 Best만 이어받습니다. 전체 train 학습도 같은 크기의 원본 baseline과 실효 파라미터를 대조합니다.

In [13]:
full_reference_prep = FeaturePreprocessor()
full_reference_X = full_reference_prep.fit_transform(X)
full_reference_model = fit_cat(full_reference_X, y)
final_prep = FeaturePreprocessor(candidate_features)
X_full_model = final_prep.fit_transform(X)
submission_model = fit_cat(X_full_model, y)
assert submission_model.get_all_params() == full_reference_model.get_all_params()
print('이 버전 CSV의 피처:', X_full_model.columns.tolist())
print('최종 학습 실효 파라미터:', submission_model.get_all_params())

이 버전 CSV의 피처: ['pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'gender_female', 'gender_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Title_Rev']
최종 학습 실효 파라미터: {'nan_mode': 'Min', 'eval_metric': 'Logloss', 'iterations': 1000, 'sampling_frequency': 'PerTree', 'leaf_estimation_method': 'Newton', 'random_score_type': 'NormalWithModelSizeDecrease', 'grow_policy': 'SymmetricTree', 'penalties_coefficient': 1, 'boosting_type': 'Plain', 'model_shrink_mode': 'Constant', 'feature_border_type': 'GreedyLogSum', 'bayesian_matrix_reg': 0.10000000149011612, 'eval_fraction': 0, 'force_unit_auto_pair_weights': False, 'l2_leaf_reg': 3, 'random_strength': 1, 'rsm': 1, 'boost_from_average': False, 'model_size_reg': 0.5, 'pool_metainfo_options': {'tags': {}}, 'subsample': 0.800000011920929, 'use_best_model': False, 'class_names': [0, 1], 'random_seed': 42, 'depth': 6, 'posterior_sampling': False, 'border_count': 2

# 10. Test 예측

In [14]:
X_test_model = final_prep.transform(X_test_raw)
assert X_test_model.columns.equals(X_full_model.columns)
positive_index = list(submission_model.classes_).index(1)
predictions = submission_model.predict_proba(X_test_model)[:, positive_index]
assert np.isfinite(predictions).all()
assert ((predictions>=0)&(predictions<=1)).all()

# 11. Submission 생성
템플릿 예측 열만 교체하고 원래 ID/순서를 유지합니다. 기존 버전 덮어쓰기를 방지합니다.

In [15]:
output_path = Path('submission/titanic_result_3.csv')
assert not output_path.exists(), '기존 결과를 덮어쓰지 않습니다.'
assert submission.columns.tolist()==[id_col,target_col]
assert len(submission)==len(test)
assert submission[id_col].is_unique and submission[id_col].notna().all()
assert set(submission[id_col])==set(test[id_col])
result = submission.copy(deep=True)
by_id = pd.Series(predictions,index=test[id_col].to_numpy())
result[target_col] = result[id_col].map(by_id)
assert result[target_col].notna().all()
result.to_csv(output_path,index=False)
saved = pd.read_csv(output_path)
assert saved.shape==submission.shape==(len(test),2)
assert saved.columns.equals(submission.columns)
pd.testing.assert_frame_equal(saved.drop(columns=target_col),submission.drop(columns=target_col))
assert pd.api.types.is_float_dtype(saved[target_col])
assert saved[target_col].notna().all() and saved[target_col].between(0,1).all()
np.testing.assert_allclose(saved[target_col],by_id.loc[submission[id_col]],atol=1e-15,rtol=1e-12)
print('검증 통과:', output_path, saved.shape)

검증 통과: submission\titanic_result_3.csv (393, 2)


## 실제 EDA 관측과 판단
학습 표본에서 Mr 393명 생존율 10.2%, Mrs 100명 89.0%, Miss 139명 82.7%, Master 36명 33.3%, Rare 19명 21.1%다. 호칭별 차이가 크지만 성별/연령 정보와 겹친다. 단변수 생존율 차이가 크다는 사실이 모델의 독립적인 추가 기여를 보장하지 않는다.

# 12. 해당 버전 실험 결과

가설: 이름 원문은 고차원이지만 호칭은 성별·연령과 관련된 낮은 차원의 정보를 보존할 수 있다.

| 항목 | 이전 Best | 이번 실험 |
|---|---:|---:|
| train_auc | 0.963079 | 0.965168 |
| validation_auc | 0.900065 | 0.902992 |
| gap | 0.063014 | 0.062176 |
| CV mean | 0.901775 | 0.900010 |
| CV std | 0.016322 | 0.016136 |

**제외**: CV 평균 또는 Gap 기준 미충족. 이전 Best 대비 +0.002927, 최초 baseline 대비 +0.002927.

다음 버전의 추가 피처: 없음.
이번 CSV 입력: ['Title'].

기존 피처와의 관계: gender·age와 정보가 겹치므로 성별 교차표와 호칭별 연령 통계를 함께 확인한다. 원본 성별/나이만으로 설명되지 않는 정보가 있는지는 AUC로 판단한다. Rare의 생존율은 작은 표본을 합친 값이다.
개별 집단 비율은 EDA 표에 있으며 소표본/인과관계 해석에 주의합니다. CV 미실행은 0점이 아니라 holdout에서 우선 제외했음을 의미합니다.
